1. EDA — ALL 3 FILES
   ├── patient.csv          ✅
   ├── vitalperiodic.csv    ✅
   └── lab.csv              ✅

                ↓

2. Define ML Problem
   ├── Target = hospitaldischargestatus
   ├── Prediction point = ICU admission
   └── Prediction window = first 24 hours

                ↓

3. Data Cleaning
   ├── Handle missing values
   ├── Handle invalid values
   ├── Handle duplicates
   ├── Handle outliers
   └── Handle data types

                ↓

4. Temporal Filtering
   ├── Vitals: 0–1440 minutes
   └── Labs:   0–1440 minutes

                ↓

5. Feature Engineering
   ├── Patient features
   ├── Vital features
   │     ├── mean
   │     ├── min
   │     ├── max
   │     └── std
   │
   └── Lab features
         ├── mean
         ├── min
         ├── max
         └── std

                ↓

6. Aggregation
   ├── Vital → ONE ROW / ICU STAY
   └── Lab   → ONE ROW / ICU STAY

                ↓

7. Combine 3 Files
   └── Merge using patientunitstayid

                ↓

8. Final Dataset Validation
   ├── Shape
   ├── Missing values
   ├── Duplicates
   ├── Target distribution
   └── Leakage check

                ↓

9. Train / Validation / Test Split
   └── Split BEFORE fitting preprocessing

                ↓

10. Preprocessing Pipeline
    ├── Imputation
    ├── Encoding
    ├── Scaling (if required)
    └── Feature selection

                ↓

11. Model Training
    ├── Baseline
    ├── Logistic Regression
    ├── Random Forest
    ├── XGBoost / LightGBM
    └── Compare models

                ↓

12. Model Evaluation
    ├── Accuracy
    ├── Precision
    ├── Recall
    ├── F1
    ├── ROC-AUC
    └── Confusion Matrix

                ↓

13. Class Imbalance Handling
    ├── class_weight
    ├── threshold tuning
    └── possibly SMOTE
       (ONLY on training data)

                ↓

14. Explainability
    └── SHAP

                ↓

15. Final Model
    ├── Save model
    ├── Prediction function
    └── API/deployment

04_preprocessing.ipynb

1. Load all 3 datasets

2. Define target
   hospitaldischargestatus

3. Define prediction window
   First 24 hours

4. Patient preprocessing
   ├── cleaning
   ├── missing values
   └── leakage columns

5. Vital preprocessing
   ├── 0–1440 minutes
   ├── invalid values
   └── aggregation

6. Lab preprocessing
   ├── 0–1440 minutes
   ├── invalid values
   └── aggregation

7. Merge
   └── patientunitstayid

8. Final dataset

9. Final validation
   ├── shape
   ├── missing values
   ├── duplicates
   └── target balance

10. Save final dataset

In [104]:
import pandas as pd
import numpy as np
patient = pd.read_csv(r"C:\2)Projects\ICU_Patient_Predict_Risk_Analysis\dataset\patient.csv")
vitalperiodic = pd.read_csv(r"C:\2)Projects\ICU_Patient_Predict_Risk_Analysis\dataset\vitalPeriodic.csv")
lab = pd.read_csv(r"C:\2)Projects\ICU_Patient_Predict_Risk_Analysis\dataset\lab.csv")

print("Patient:", patient.shape)
print("Vital:", vitalperiodic.shape)
print("Lab:", lab.shape)

Patient: (2520, 29)
Vital: (1634960, 19)
Lab: (434660, 10)


In [105]:
# step 3: Define a TARGET

target = "hospitaldischargestatus"

print(patient[target].value_counts(dropna=False))

hospitaldischargestatus
Alive      2280
Expired     212
NaN          28
Name: count, dtype: int64


In [106]:
#Step 4 — Define prediction window

# we decided

#Prediction point = ICU admission
# Prediction window = first 24 hours

#   24 hours is :     24 × 60 = 1440 minutes


WINDOW = 1440

'''
Our rule:

0 ≤ offset ≤ 1440

This is very important because it prevents using future information.


'''

'\nOur rule:\n\n0 ≤ offset ≤ 1440\n\nThis is very important because it prevents using future information.\n\n\n'

In [107]:
#step 5 :  patieint pre processing:

#   Patient data is already approximately one row per ICU stay.

#  First create a copy:


patient_prep = patient.copy()

# Step 5.1 — Remove target missing rows

patient_prep = patient_prep[patient_prep[target].notna()].copy()

print(patient_prep[target].isna().sum())

0


In [108]:
# Step 6 — Remove obvious leakage columns from patient data

'''
This is very important.

We are predicting:

hospitaldischargestatus

Therefore information that is only known at/after discharge should not be used as input.

Start with:


'''


patient_leakage_cols = [
    "hospitaldischargestatus",
    "hospitaldischargelocation",
    "hospitaldischargeoffset",
    "unitdischargestatus",
    "unitdischargelocation",
    "unitdischargeoffset",
    "dischargeweight"
]


# remove them

patient_features = patient_prep.drop(
    columns=patient_leakage_cols,
    errors="ignore"
)

# keep target seperately
patient_target = patient_prep[["patientunitstayid", target]].copy()

In [109]:
#step 7 : hnadles msiing vlaue AGE
'''
 Your patient data contains: > 89
So convert it to numeric:
'''
patient_features["age"] = pd.to_numeric(patient_features["age"],errors="coerce")

print(patient_features["age"].values)

[87. 87. 76. ... 41. 72. 50.]


In [110]:
print('>89' in patient_features["age"].values)

False


#Step 8 — Vital preprocessing

In [111]:
# Step 8 — Vital preprocessing

# Create a copy:

vital_prep = vitalperiodic.copy()


#   Step 8.1 — Select first 24 hours

vital_prep = vital_prep[
    (vital_prep["observationoffset"] >= 0) &
    (vital_prep["observationoffset"] <= WINDOW)
].copy()

# 
print("vital_prep shape ",vital_prep.shape)
print("vital_prep - observationoffset minimum value ",vital_prep["observationoffset"].min())
print("vital_prep - observationoffset Maximum value",vital_prep["observationoffset"].max())

vital_prep shape  (553846, 19)
vital_prep - observationoffset minimum value  0
vital_prep - observationoffset Maximum value 1440


In [112]:
#  STEP 9 : Clinical features selection for 16 clinical measurement columns for vital-sign aggregation.

# vital ID/time columns are not required for  aggregation


vital_feature_cols = [
    "temperature",
    "sao2",
    "heartrate",
    "respiration",
    "cvp",
    "etco2",
    "systemicsystolic",
    "systemicdiastolic",
    "systemicmean",
    "pasystolic",
    "padiastolic",
    "pamean",
    "st1",
    "st2",
    "st3",
    "icp"
]

In [113]:
# Step 10 — Aggregate Vital data

'''
This is the key transformation.

Before:

ICU 1001 → many rows
ICU 1001 → many rows
ICU 1001 → many rows

After:

ICU 1001 → ONE ROW

We'll calculate:

mean
min
max
standard deviation 

'''
vital_agg = (vital_prep.groupby("patientunitstayid")[vital_feature_cols].agg(["mean", "min", "max", "std"]))

# flatten the columnnames


vital_agg.columns = [
    f"{col}_{stat}"
    for col, stat in vital_agg.columns
]

# convert index back to columns

vital_agg = vital_agg.reset_index()

print(vital_agg.shape)
vital_agg.head()

(2370, 65)


,patientunitstayid,temperature_mean,temperature_min,temperature_max,temperature_std,sao2_mean,sao2_min,sao2_max,sao2_std,heartrate_mean,...,st2_max,st2_std,st3_mean,st3_min,st3_max,st3_std,icp_mean,icp_min,icp_max,icp_std
0,141764,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,106.652174,...,-0.7,0.225750,-0.818182,-1.7,-0.5,0.197288,NaN,NaN,NaN,NaN
1,141765,NaN,NaN,NaN,NaN,96.608696,95.0,98.0,0.687278,83.190972,...,-0.5,0.189611,-0.613636,-0.9,-0.4,0.148950,NaN,NaN,NaN,NaN
2,143870,NaN,NaN,NaN,NaN,96.284810,80.0,100.0,2.243438,45.449367,...,0.2,0.048208,0.124684,0.0,0.3,0.051335,NaN,NaN,NaN,NaN
3,144815,NaN,NaN,NaN,NaN,98.630682,96.0,100.0,0.878292,83.800000,...,0.2,0.120471,-0.053636,-0.9,0.0,0.096564,NaN,NaN,NaN,NaN
4,145427,NaN,NaN,NaN,NaN,95.761905,89.0,100.0,1.989688,65.457983,...,0.5,0.134240,0.087605,-0.2,0.4,0.130738,NaN,NaN,NaN,NaN


note :  now we got  1 ICU stay → 1 row

In [114]:
print("Before aggregation:", vital_prep.shape)
print("After aggregation:", vital_agg.shape)
print("Unique ICU stays:", vital_prep["patientunitstayid"].nunique())

Before aggregation: (553846, 19)
After aggregation: (2370, 65)
Unique ICU stays: 2370


#Step 11 — Lab preprocessing


In [115]:
lab_prep = lab.copy()
# Step 11.1 — Select first 24 hours

lab_prep = lab_prep[
    (lab_prep["labresultoffset"] >= 0) &
    (lab_prep["labresultoffset"] <= WINDOW)
].copy()


print(" lab preproceesing shape ",lab_prep.shape)
print(" lab preproceesing min value ",lab_prep["labresultoffset"].min())
print(" lab preproceesing max value ",lab_prep["labresultoffset"].max())

 lab preproceesing shape  (98870, 10)
 lab preproceesing min value  0
 lab preproceesing max value  1440


In [116]:
# Step 12 — Remove missing lab results

# We cannot calculate numerical features from missing labresult.

lab_prep = lab_prep[lab_prep["labresult"].notna()].copy()
print(lab_prep.head())

        labid  patientunitstayid  labresultoffset  labtypeid     labname  \
20  415732508            1754323              603          1     glucose   
24  415732507            1754323              603          1     calcium   
28  415732503            1754323              603          1   potassium   
32  415732504            1754323              603          1  creatinine   
37  415732505            1754323              603          1   anion gap   

    labresult labresulttext labmeasurenamesystem labmeasurenameinterface  \
20     116.00         116                  mg/dL                   mg/dL   
24       7.90           7.9                mg/dL                   mg/dL   
28       4.20           4.2               mmol/L                  mmol/L   
32       0.94          0.94                mg/dL                   mg/dL   
37       6.00             6                  NaN                  mmol/L   

    labresultrevisedoffset  
20                     667  
24                     667  

In [117]:
# Step 13 — Lab aggregation
'''
This is slightly different from Vital data

# We have:

labname
labresult

For example:

ICU 1001
    glucose       120
    glucose       130
    sodium        140
    sodium        138
    potassium       4.2

We want:

ICU 1001

glucose_mean
glucose_min
glucose_max
glucose_std

sodium_mean
sodium_min
sodium_max
sodium_std

potassium_mean
...

'''

'\nThis is slightly different from Vital data\n\n# We have:\n\nlabname\nlabresult\n\nFor example:\n\nICU 1001\n    glucose       120\n    glucose       130\n    sodium        140\n    sodium        138\n    potassium       4.2\n\nWe want:\n\nICU 1001\n\nglucose_mean\nglucose_min\nglucose_max\nglucose_std\n\nsodium_mean\nsodium_min\nsodium_max\nsodium_std\n\npotassium_mean\n...\n\n'

In [118]:
lab_agg = (
    lab_prep
    .groupby(["patientunitstayid", "labname"])["labresult"]
    .agg(["mean", "min", "max", "std"])
    .unstack()
)

# Flatten the columns

lab_agg.columns = [
    f"{labname}_{stat}"
    for stat, labname in lab_agg.columns
]

# reset Index

lab_agg = lab_agg.reset_index()

# verfiy

print(lab_agg.shape)
lab_agg.head()


(2339, 525)


,patientunitstayid,-bands_mean,-basos_mean,-eos_mean,-lymphs_mean,-monos_mean,-polys_mean,24 h urine protein_mean,ALT (SGPT)_mean,AST (SGOT)_mean,...,total protein_std,transferrin_std,triglycerides_std,troponin - I_std,troponin - T_std,uric acid_std,urinary creatinine_std,urinary osmolality_std,urinary sodium_std,urinary specific gravity_std
0,141765,NaN,0.0,1.0,19.0,10.0,70.0,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,143870,NaN,0.0,0.0,8.0,4.0,88.0,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,144815,NaN,1.0,1.0,24.0,8.0,66.0,NaN,42.0,39.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,145427,NaN,0.0,0.0,8.0,8.0,84.0,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,147306,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [119]:
# now one row , one ICU Stay

In [120]:
#  Step 14 — Combine Vital + Lab

vital_lab = pd.merge(
    vital_agg,
    lab_agg,
    on="patientunitstayid",
    how="outer"
)

print(vital_lab.shape)

(2454, 589)


In [121]:
#Step 15 — Combine with Patient data

# Now merge the patient features and target:

final_df = pd.merge(
    patient_features,
    vital_lab,
    on="patientunitstayid",
    how="inner"
)

# Then add target:

final_df = pd.merge(
    final_df,
    patient_target,
    on="patientunitstayid",
    how="inner"
)

# Check:

print(final_df.shape)

print("final df shape rows and coluns are 2426 and 211 columns")

(2426, 611)
final df shape rows and coluns are 2426 and 211 columns


Good. (2426, 611) is a reasonable result.

It means:

Result	Meaning
2,426 rows	2,426 ICU stays successfully matched across the 3 datasets
611 columns	Patient + aggregated Vital + aggregated Lab features
1 row	Represents 1 ICU stay

Remember, we started with:

Patient       → ~2,520 ICU stays
Vital         → 2,375 ICU stays
Lab           → 2,444 ICU stays

After the merge, 2,426 ICU stays remained because we used an inner merge between patient features and the Vital+Lab data.

In [122]:
# Step 16 — Check duplicate ICU stays

# This is extremely important point 
# we want 1 ICU stay = 1 row

print(
    "Duplicate ICU stays:",
    final_df["patientunitstayid"].duplicated().sum()
)

Duplicate ICU stays: 0


In [123]:
# Step 17 — Check target
print(final_df[target].value_counts(dropna=False))

# Check percentages:

print(
    final_df[target]
    .value_counts(normalize=True)
    .mul(100)
)

hospitaldischargestatus
Alive      2218
Expired     208
Name: count, dtype: int64
hospitaldischargestatus
Alive      91.426216
Expired     8.573784
Name: proportion, dtype: float64


In [124]:
#Step 18 — Check missing values
missing = (
    final_df.isnull()
    .sum()
    .sort_values(ascending=False)
)

missing = missing[missing > 0]

print(missing.head(20))

#Do not fill everything with mean/median yet.

#We first need to understand the missingness after merging.

protein C_std                       2426
protein - CSF_std                   2426
prolactin_std                       2426
TIBC_std                            2426
prealbumin_std                      2426
glucose - CSF_std                   2426
haptoglobin_std                     2426
folate_std                          2426
cd 4_std                            2426
T3_std                              2426
T3RU_std                            2426
Tobramycin - peak_std               2426
Tacrolimus-FK506_std                2426
T4_std                              2426
Tobramycin - random_std             2426
Vancomycin - trough_std             2426
Tobramycin - trough_std             2426
WBC's in body fluid_std             2426
WBC's in cerebrospinal fluid_std    2426
Digoxin_std                         2426
dtype: int64


In [125]:
# Step 19 — Check final dataset
print("Final shape:", final_df.shape)

print(
    "Unique ICU stays:",
    final_df["patientunitstayid"].nunique()
)

print(
    "Duplicate rows:",
    final_df.duplicated().sum()
)

print(
    "Duplicate ICU IDs:",
    final_df["patientunitstayid"].duplicated().sum()
)

Final shape: (2426, 611)
Unique ICU stays: 2426


Duplicate rows: 0
Duplicate ICU IDs: 0


In [126]:
# Step 20 — Save the intermediate final dataset
final_df.to_csv(
    "final_preprocessed_raw.csv",
    index=False
)

In [127]:
# Step 21 — Identify target and features
target = "hospitaldischargestatus"

print(final_df[target].value_counts(dropna=False))

hospitaldischargestatus
Alive      2218
Expired     208
Name: count, dtype: int64


In [128]:
# Step 22 — Remove target leakage / unwanted columns
'''
We already removed the obvious discharge columns from patient.csv,
but let's check the final dataset for columns that should not become model features.

'''

print(final_df.columns.tolist())

['patientunitstayid', 'patienthealthsystemstayid', 'gender', 'age', 'ethnicity', 'hospitalid', 'wardid', 'apacheadmissiondx', 'admissionheight', 'hospitaladmittime24', 'hospitaladmitoffset', 'hospitaladmitsource', 'hospitaldischargeyear', 'hospitaldischargetime24', 'unittype', 'unitadmittime24', 'unitadmitsource', 'unitvisitnumber', 'unitstaytype', 'admissionweight', 'unitdischargetime24', 'uniquepid', 'temperature_mean', 'temperature_min', 'temperature_max', 'temperature_std', 'sao2_mean', 'sao2_min', 'sao2_max', 'sao2_std', 'heartrate_mean', 'heartrate_min', 'heartrate_max', 'heartrate_std', 'respiration_mean', 'respiration_min', 'respiration_max', 'respiration_std', 'cvp_mean', 'cvp_min', 'cvp_max', 'cvp_std', 'etco2_mean', 'etco2_min', 'etco2_max', 'etco2_std', 'systemicsystolic_mean', 'systemicsystolic_min', 'systemicsystolic_max', 'systemicsystolic_std', 'systemicdiastolic_mean', 'systemicdiastolic_min', 'systemicdiastolic_max', 'systemicdiastolic_std', 'systemicmean_mean', 'syst

In [129]:
# For this project, remove the ID:

'''
Why?

patientunitstayid is only an identifier. It has no clinical meaning.
'''

final_df = final_df.drop(
    columns=["patientunitstayid"],
    errors="ignore"
)

In [130]:
# Step 23 — Convert target to 0/1

print(final_df[target].value_counts())

final_df[target] = final_df[target].map({
    "Alive": 0,
    "Expired": 1
})
# check

print(final_df[target].value_counts(dropna=False))

hospitaldischargestatus
Alive      2218
Expired     208
Name: count, dtype: int64
hospitaldischargestatus
0    2218
1     208
Name: count, dtype: int64


In [131]:
# Step 24 — Separate X and y

X = final_df.drop(columns=[target])

y = final_df[target]

print("X shape:", X.shape)
print("y shape:", y.shape)

X shape: (2426, 609)
y shape: (2426,)


In [132]:
# Step 25 — Check feature data types

print(X.dtypes.value_counts())

float64    591
object      12
int64        6
Name: count, dtype: int64


In [133]:
categorical_cols = X.select_dtypes(
    include=["object"]
).columns.tolist()

print("Categorical columns:")
print(len(categorical_cols))

Categorical columns:
12


In [134]:
numerical_cols = X.select_dtypes(
    include=["int64", "float64"]
).columns.tolist()

print("Number of numerical columns:",
      len(numerical_cols))

Number of numerical columns: 597


In [135]:
# Step 26 — Check missing values in X

missing = X.isnull().sum()

missing = missing[missing > 0].sort_values(
    ascending=False
)
print(missing.count())
print(missing.head(3))

596
protein C_std        2426
protein S_std        2426
protein - CSF_std    2426
dtype: int64


NOTE : This is an important step because your Vital and Lab data naturally has many missing values.

Do NOT simply drop all rows.

For example:

ICP missing

doesn't necessarily mean the patient record is bad.

It may mean:

ICP was not measured for that ICU patient.

We'll handle this appropriately with imputation.

In [136]:
# Step 27 — Train/Test split

#  This step must happen before fitting the imputer, encoder, or scaler.


'''

In Step 27, we use stratify=y because your target hospitaldischargestatus is imbalanced.

For example, from your patient.csv EDA you had approximately:

Target	Count
Alive	2280
Expired	212

So roughly:

Alive    → 91.5%
Expired  → 8.5%

That means the classes are not equally represented.

Why stratify=y?

When we split:

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

stratify=y tries to maintain the same Alive/Expired proportion in both train and test.

Example:

Original
Alive    91.5%
Expired   8.5%

       ↓ stratified split

Training
Alive    ~91.5%
Expired  ~8.5%

Testing
Alive    ~91.5%
Expired  ~8.5%
Important

stratify=y does NOT fix the class imbalance.

It only makes sure the train/test split preserves the imbalance fairly.

Actual imbalance handling comes later, during model training, using things like class_weight,
 threshold tuning, or possibly SMOTE.

'''

'\n\nIn Step 27, we use stratify=y because your target hospitaldischargestatus is imbalanced.\n\nFor example, from your patient.csv EDA you had approximately:\n\nTarget\tCount\nAlive\t2280\nExpired\t212\n\nSo roughly:\n\nAlive    → 91.5%\nExpired  → 8.5%\n\nThat means the classes are not equally represented.\n\nWhy stratify=y?\n\nWhen we split:\n\nX_train, X_test, y_train, y_test = train_test_split(\n    X,\n    y,\n    test_size=0.2,\n    random_state=42,\n    stratify=y\n)\n\nstratify=y tries to maintain the same Alive/Expired proportion in both train and test.\n\nExample:\n\nOriginal\nAlive    91.5%\nExpired   8.5%\n\n       ↓ stratified split\n\nTraining\nAlive    ~91.5%\nExpired  ~8.5%\n\nTesting\nAlive    ~91.5%\nExpired  ~8.5%\nImportant\n\nstratify=y does NOT fix the class imbalance.\n\nIt only makes sure the train/test split preserves the imbalance fairly.\n\nActual imbalance handling comes later, during model training, using things like class_weight,\n threshold tuning, or po

In [137]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("X_train:", X_train.shape)
print("X_test :", X_test.shape)

print("y_train:")
print(y_train.value_counts())

print("y_test:")
print(y_test.value_counts())

X_train: (1940, 609)
X_test : (486, 609)
y_train:
hospitaldischargestatus
0    1774
1     166
Name: count, dtype: int64
y_test:
hospitaldischargestatus
0    444
1     42
Name: count, dtype: int64


In [138]:
# Step 28 — Create preprocessing pipeline

'''
Now we need to handle:


Numerical columns
      ↓
Missing values
      ↓
Scaling

Categorical columns
      ↓
Missing values
      ↓
One-hot encoding

'''

'\nNow we need to handle:\n\n\nNumerical columns\n      ↓\nMissing values\n      ↓\nScaling\n\nCategorical columns\n      ↓\nMissing values\n      ↓\nOne-hot encoding\n\n'

In [139]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])



categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(
        handle_unknown="ignore"
    ))
])


preprocessor = ColumnTransformer([
    ("num", numeric_pipeline, numerical_cols),
    ("cat", categorical_pipeline, categorical_cols)
])

In [140]:
# Step 29 — Fit preprocessing ONLY on training data

X_train_processed = preprocessor.fit_transform(X_train)

X_test_processed = preprocessor.transform(X_test)


c:\2)Projects\ICU_Patient_Predict_Risk_Analysis\.venv\Lib\site-packages\sklearn\impute\_base.py:641: UserWarning: Skipping features without any observed values: ['Tobramycin - random_mean' 'prolactin_mean' 'Tobramycin - random_min'
 'prolactin_min' 'Tobramycin - random_max' 'prolactin_max'
 '24 h urine protein_std' 'CRP-hs_std' 'Digoxin_std'
 'Gentamicin - random_std' 'T3_std' 'T3RU_std' 'T4_std' 'TIBC_std'
 'Tacrolimus-FK506_std' 'Tobramycin - peak_std' 'Tobramycin - random_std'
 'Tobramycin - trough_std' 'Vancomycin - trough_std'
 "WBC's in body fluid_std" "WBC's in cerebrospinal fluid_std" 'cd 4_std'
 'folate_std' 'glucose - CSF_std' 'haptoglobin_std' 'prealbumin_std'
 'prolactin_std' 'protein - CSF_std' 'protein C_std' 'protein S_std'
 'reticulocyte count_std' 'uric acid_std']. At least one non-missing value is needed for imputation with strategy='median'.
  warnings.warn(
c:\2)Projects\ICU_Patient_Predict_Risk_Analysis\.venv\Lib\site-packages\sklearn\impute\_base.py:641: UserWarni

In [141]:
# step 30 : checked process data

print("Processed X_train shape:",
      X_train_processed.shape)

print("Processed X_test shape:",
      X_test_processed.shape)

Processed X_train shape: (1940, 6034)
Processed X_test shape: (486, 6034)


In [142]:

'''
NOTE:

-> How 6034 columns came

PATIENT.CSV
2520 rows
        │
        │
        ├──────────────┐
        │              │
VITALPERIODIC.CSV   LAB.CSV
1,634,960 rows      434,660 rows
        │              │
        ↓              ↓
 First 24 hours    First 24 hours
        │              │
        ↓              ↓
 Aggregation       Aggregation
        │              │
        ↓              ↓
 ~2375 stays       ~2444 stays
        │              │
        └──────┬───────┘
               ↓
          MERGE ALL 3
               ↓
      final_df = (2426, 611)
               │
               ↓
       Remove ID + target
               ↓
             X, y
               │
               ↓
        Train/Test Split
          /          \
         ↓            ↓
 X_train (1940, ~609)  X_test (486, ~609)
         │
         ↓
    Preprocessing
    ├── Imputation
    ├── Scaling
    └── OneHotEncoding
         │
         ↓
X_train_processed = (1940, 6034)
X_test_processed  = (486, 6034)

'''

'\nNOTE:\n\n-> How 6034 columns came\n\nPATIENT.CSV\n2520 rows\n        │\n        │\n        ├──────────────┐\n        │              │\nVITALPERIODIC.CSV   LAB.CSV\n1,634,960 rows      434,660 rows\n        │              │\n        ↓              ↓\n First 24 hours    First 24 hours\n        │              │\n        ↓              ↓\n Aggregation       Aggregation\n        │              │\n        ↓              ↓\n ~2375 stays       ~2444 stays\n        │              │\n        └──────┬───────┘\n               ↓\n          MERGE ALL 3\n               ↓\n      final_df = (2426, 611)\n               │\n               ↓\n       Remove ID + target\n               ↓\n             X, y\n               │\n               ↓\n        Train/Test Split\n          /                   ↓            ↓\n X_train (1940, ~609)  X_test (486, ~609)\n         │\n         ↓\n    Preprocessing\n    ├── Imputation\n    ├── Scaling\n    └── OneHotEncoding\n         │\n         ↓\nX_train_processed = (194

In [143]:
# Step 31 — Check the 6,034 features

#Find out why 609-ish features became 6,034 and identify high-cardinality categorical columns.

print("Original numerical columns:", len(numerical_cols))
print("Original categorical columns:", len(categorical_cols))

for col in categorical_cols:
    print(col, X_train[col].nunique())

Original numerical columns: 597
Original categorical columns: 12
gender 2
ethnicity 6
apacheadmissiondx 219
hospitaladmittime24 1154
hospitaladmitsource 13
hospitaldischargetime24 646
unittype 8
unitadmittime24 1028
unitadmitsource 13
unitstaytype 4
unitdischargetime24 873
uniquepid 1503


In [144]:
# Step 32 — Remove these columns

high_cardinality_cols = [
    "hospitaladmittime24",
    "hospitaldischargetime24",
    "unitadmittime24",
    "unitdischargetime24",
    "uniquepid"
]

X = X.drop(columns=high_cardinality_cols, errors="ignore")
print(X.shape)

(2426, 604)


In [145]:
'''
That means we removed 5 columns:

609 → 604 features

'''

'\nThat means we removed 5 columns:\n\n609 → 604 features\n\n'

In [146]:
'''
Step 33 — Redo the train/test split

Because X has changed, our old X_train and X_test are based on the old 609 features. So we need to recreate them.

Run only this:

'''

'\nStep 33 — Redo the train/test split\n\nBecause X has changed, our old X_train and X_test are based on the old 609 features. So we need to recreate them.\n\nRun only this:\n\n'

In [147]:
# Step 33 — Redo the train/test split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("X_train shape:", X_train.shape)
print("X_test shape:", X_test.shape)

X_train shape: (1940, 604)
X_test shape: (486, 604)


In [148]:
# Step 34 — Identify numerical and categorical columns again
'''
Because we changed X, we need to recreate these lists.

Run only this:

'''

categorical_cols = X_train.select_dtypes(
    include=["object"]
).columns.tolist()

numerical_cols = X_train.select_dtypes(
    include=["int64", "float64"]
).columns.tolist()

print("Numerical columns:", len(numerical_cols))
print("Categorical columns:", len(categorical_cols))

print("\nCategorical columns:")
print(categorical_cols)

Numerical columns: 597
Categorical columns: 7

Categorical columns:
['gender', 'ethnicity', 'apacheadmissiondx', 'hospitaladmitsource', 'unittype', 'unitadmitsource', 'unitstaytype']


In [149]:
# Step 35 — Check class imbalance

print("Training target distribution:")
print(y_train.value_counts())

print("\nTraining target percentage:")
print(y_train.value_counts(normalize=True) * 100)

Training target distribution:
hospitaldischargestatus
0    1774
1     166
Name: count, dtype: int64

Training target percentage:
hospitaldischargestatus
0    91.443299
1     8.556701
Name: proportion, dtype: float64



EDA finding to document

Target variable is highly imbalanced. The majority class (Alive) represents 91.44% of the training data, while the minority class (Expired) represents only 8.56%. Therefore, accuracy alone should not be used as the primary evaluation metric. Class imbalance handling and metrics such as Recall, Precision, F1-score, and ROC-AUC will be considered during model development.

One important clarification

We are not balancing the data now.

At this stage we are only identifying the imbalance.

Later, during model training, we'll handle it appropriately—for example with:

class_weight="balanced"

In [150]:
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer

numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer([
    ("num", numeric_pipeline, numerical_cols),
    ("cat", categorical_pipeline, categorical_cols)
])

X_train_processed = preprocessor.fit_transform(X_train)

X_test_processed = preprocessor.transform(X_test)

print("Processed X_train shape:", X_train_processed.shape)
print("Processed X_test shape:", X_test_processed.shape)

print("Train has NaN:", np.isnan(X_train_processed.data).sum())
print("Test has NaN :", np.isnan(X_test_processed.data).sum())

c:\2)Projects\ICU_Patient_Predict_Risk_Analysis\.venv\Lib\site-packages\sklearn\impute\_base.py:641: UserWarning: Skipping features without any observed values: ['Tobramycin - random_mean' 'prolactin_mean' 'Tobramycin - random_min'
 'prolactin_min' 'Tobramycin - random_max' 'prolactin_max'
 '24 h urine protein_std' 'CRP-hs_std' 'Digoxin_std'
 'Gentamicin - random_std' 'T3_std' 'T3RU_std' 'T4_std' 'TIBC_std'
 'Tacrolimus-FK506_std' 'Tobramycin - peak_std' 'Tobramycin - random_std'
 'Tobramycin - trough_std' 'Vancomycin - trough_std'
 "WBC's in body fluid_std" "WBC's in cerebrospinal fluid_std" 'cd 4_std'
 'folate_std' 'glucose - CSF_std' 'haptoglobin_std' 'prealbumin_std'
 'prolactin_std' 'protein - CSF_std' 'protein C_std' 'protein S_std'
 'reticulocyte count_std' 'uric acid_std']. At least one non-missing value is needed for imputation with strategy='median'.
  warnings.warn(
c:\2)Projects\ICU_Patient_Predict_Risk_Analysis\.venv\Lib\site-packages\sklearn\impute\_base.py:641: UserWarni

Processed X_train shape: (1940, 830)
Processed X_test shape: (486, 830)
Train has NaN: 0
Test has NaN : 0


In [151]:
'''
NOTE 2:
Raw combined dataset
2426 × 611
        ↓
Remove 5 unwanted columns
        ↓
2426 × 604
        ↓
Train/Test split
        ↓
1940 × 604
486 × 604
        ↓
Imputation + Scaling + One-Hot Encoding
        ↓
1940 × 830   ← final processed train
486 × 830    ← final processed test


So 830 is completely reasonable.

The warning about completely missing lab features is a separate issue. We should handle that before modeling, but your 604 → 830 increase itself is normal and expected.

-----------------------------------------------------------------------------------------------------------------------------
Detailed explanation:

Before encoding, you had:

X_train = (1940, 604)

That means:

1,940 ICU stays × 604 raw features

These 604 columns were:

597 numerical
+ 7 categorical
= 604

Then we applied preprocessing.

What happened to the 7 categorical columns?

They were One-Hot Encoded.

For example:

gender
------
Male
Female

becomes:

gender_Male
gender_Female

Similarly:

ethnicity
apacheadmissiondx
hospitaladmitsource
unittype
unitadmitsource
unitstaytype

can each have multiple categories.

So:

Before encoding:

597 numerical + 7 categorical
                  ↓
                604

After encoding:

597 numerical
+
~233 encoded categorical features
=
830 features

So:

X_train              → (1940, 604)
                         ↓
                  preprocessing
                         ↓
X_train_processed     → (1940, 830)
Very important

830 does NOT mean you suddenly got 226 new medical features.

It means the 7 categorical columns were expanded into 233 binary columns by One-Hot Encoding.

For example:

apacheadmissiondx
219 categories
       ↓
~219 binary columns

That is the main reason for the increase.

'''

'\nNOTE 2:\nRaw combined dataset\n2426 × 611\n        ↓\nRemove 5 unwanted columns\n        ↓\n2426 × 604\n        ↓\nTrain/Test split\n        ↓\n1940 × 604\n486 × 604\n        ↓\nImputation + Scaling + One-Hot Encoding\n        ↓\n1940 × 830   ← final processed train\n486 × 830    ← final processed test\n\n\nSo 830 is completely reasonable.\n\nThe warning about completely missing lab features is a separate issue. We should handle that before modeling, but your 604 → 830 increase itself is normal and expected.\n\n-----------------------------------------------------------------------------------------------------------------------------\nDetailed explanation:\n\nBefore encoding, you had:\n\nX_train = (1940, 604)\n\nThat means:\n\n1,940 ICU stays × 604 raw features\n\nThese 604 columns were:\n\n597 numerical\n+ 7 categorical\n= 604\n\nThen we applied preprocessing.\n\nWhat happened to the 7 categorical columns?\n\nThey were One-Hot Encoded.\n\nFor example:\n\ngender\n------\nMale\nFema

#MISSING FEATURES cross Check with original data ( x_train)  not on the x_train_Processed 

In [152]:
missing_all = X_train.isna().all()

print("Completely missing features:", missing_all.sum())

print(X_train.columns[missing_all].tolist())

Completely missing features: 32
['Tobramycin - random_mean', 'prolactin_mean', 'Tobramycin - random_min', 'prolactin_min', 'Tobramycin - random_max', 'prolactin_max', '24 h urine protein_std', 'CRP-hs_std', 'Digoxin_std', 'Gentamicin - random_std', 'T3_std', 'T3RU_std', 'T4_std', 'TIBC_std', 'Tacrolimus-FK506_std', 'Tobramycin - peak_std', 'Tobramycin - random_std', 'Tobramycin - trough_std', 'Vancomycin - trough_std', "WBC's in body fluid_std", "WBC's in cerebrospinal fluid_std", 'cd 4_std', 'folate_std', 'glucose - CSF_std', 'haptoglobin_std', 'prealbumin_std', 'prolactin_std', 'protein - CSF_std', 'protein C_std', 'protein S_std', 'reticulocyte count_std', 'uric acid_std']


In [153]:
# so we have to remove them

# What should we do?

#Since these 32 features contain zero information in the training data, we should remove them before preprocessing.

#NOTE:  Here we are not check with processed

missing_all = X_train.isna().all()

remove_missing_cols = X_train.columns[missing_all].tolist()

print("Removing:", len(remove_missing_cols), "completely missing features")

Removing: 32 completely missing features


In [154]:
X_train = X_train.drop(columns=remove_missing_cols)
X_test = X_test.drop(columns=remove_missing_cols)

print("X_train shape:", X_train.shape)
print("X_test shape:", X_test.shape)

X_train shape: (1940, 572)
X_test shape: (486, 572)


In [155]:
#Next step

#Now we need to recreate the numerical/categorical column lists because the feature set changed.

categorical_cols = X_train.select_dtypes(
    include=["object"]
).columns.tolist()

numerical_cols = X_train.select_dtypes(
    include=["int64", "float64"]
).columns.tolist()

print("Numerical columns:", len(numerical_cols))
print("Categorical columns:", len(categorical_cols))
print("Total:", len(numerical_cols) + len(categorical_cols))

Numerical columns: 565
Categorical columns: 7
Total: 572


In [156]:
preprocessor = ColumnTransformer([
    ("num", numeric_pipeline, numerical_cols),
    ("cat", categorical_pipeline, categorical_cols)
])

X_train_processed = preprocessor.fit_transform(X_train)
X_test_processed = preprocessor.transform(X_test)

print("X_train_processed shape:", X_train_processed.shape)
print("X_test_processed shape:", X_test_processed.shape)

print("Train NaN:", np.isnan(X_train_processed).sum())
print("Test NaN :", np.isnan(X_test_processed).sum())

X_train_processed shape: (1940, 830)
X_test_processed shape: (486, 830)
Train NaN: 0
Test NaN : 0


In [157]:
'''
NOTE:
 No warning and no NaNs, so the preprocessing pipeline is working.

One important correction from what I said earlier: 830 is expected, not necessarily 798. The 32 removed columns were all-missing numeric columns, but the current pipeline's encoded output confirms the final transformed feature space is 830.

So your current state is:


| Stage                                  |                   Shape |
| -------------------------------------- | ----------------------: |
| Combined dataset                       |           `(2426, 611)` |
| After removing unwanted columns        |           `(2426, 604)` |
| After removing 32 all-missing features |           `(2426, 572)` |


| Train                                  |           `(1940, 572)` |
| Test                                   |            `(486, 572)` |


| After preprocessing/encoding           | **Train `(1940, 830)`** |
|                                        |   **Test `(486, 830)`** |
| NaN after preprocessing                |                 **0** ✅ |


'''

"\nNOTE:\n No warning and no NaNs, so the preprocessing pipeline is working.\n\nOne important correction from what I said earlier: 830 is expected, not necessarily 798. The 32 removed columns were all-missing numeric columns, but the current pipeline's encoded output confirms the final transformed feature space is 830.\n\nSo your current state is:\n\n\n| Stage                                  |                   Shape |\n| -------------------------------------- | ----------------------: |\n| Combined dataset                       |           `(2426, 611)` |\n| After removing unwanted columns        |           `(2426, 604)` |\n| After removing 32 all-missing features |           `(2426, 572)` |\n\n\n| Train                                  |           `(1940, 572)` |\n| Test                                   |            `(486, 572)` |\n\n\n| After preprocessing/encoding           | **Train `(1940, 830)`** |\n|                                        |   **Test `(486, 830)`** |\n| NaN a

In [158]:
'''
NOTE: Last processing step : Checking Mortality Column



Reason for checking:

-> Before training the model, we should check for leakage one final time
  especially the mortality column.

-> The intention is not just to find the word mortality. main reason is

    " Check whether any feature in X_train contains information that directly reveals the target hospitaldischargestatus."



In this project, we already identified some leakage columns and removed them here:

patient_leakage_cols = [
    "hospitaldischargestatus",
    "hospitaldischargelocation",
    "hospitaldischargeoffset",
    "unitdischargestatus",
    "unitdischargelocation",
    "unitdischargeoffset",
    "dischargeweight"
]

And later we removed these unnecessary/high-cardinality columns:

remove_cols = [
    "hospitaladmittime24",
    "hospitaldischargetime24",
    "unitadmittime24",
    "unitdischargetime24",
    "uniquepid"
]
What we need to check now

The important question is whether mortality is also an outcome-derived feature.

Run:

print("mortality" in X_train.columns)

If it returns:

True

then we need to investigate/remove it because it may leak the target.

If:

False

then it is already not present in X_train.

So yes — leakage checking is about the meaning/source of the feature, not simply searching for a particular word.

'''

'\nNOTE: Last processing step : Checking Mortality Column\n\n\n\nReason for checking:\n\n-> Before training the model, we should check for leakage one final time\n  especially the mortality column.\n\n-> The intention is not just to find the word mortality. main reason is\n\n    " Check whether any feature in X_train contains information that directly reveals the target hospitaldischargestatus."\n\n\n\nIn this project, we already identified some leakage columns and removed them here:\n\npatient_leakage_cols = [\n    "hospitaldischargestatus",\n    "hospitaldischargelocation",\n    "hospitaldischargeoffset",\n    "unitdischargestatus",\n    "unitdischargelocation",\n    "unitdischargeoffset",\n    "dischargeweight"\n]\n\nAnd later we removed these unnecessary/high-cardinality columns:\n\nremove_cols = [\n    "hospitaladmittime24",\n    "hospitaldischargetime24",\n    "unitadmittime24",\n    "unitdischargetime24",\n    "uniquepid"\n]\nWhat we need to check now\n\nThe important question i

In [159]:

mortality_cols = []

for col in X_train.columns:
    if "mortality" in col.lower():
        mortality_cols.append(col)

print(mortality_cols)



[]


First, let's check the final target distribution once:

In [160]:
print("y_train and y_test value counts ")

print(y_train.value_counts())
print(y_test.value_counts())

print("y_train and y_test value counts percentage:")

print(y_train.value_counts(normalize=True) * 100)
print(y_test.value_counts(normalize=True) * 100)

y_train and y_test value counts 
hospitaldischargestatus
0    1774
1     166
Name: count, dtype: int64
hospitaldischargestatus
0    444
1     42
Name: count, dtype: int64
y_train and y_test value counts percentage:
hospitaldischargestatus
0    91.443299
1     8.556701
Name: proportion, dtype: float64
hospitaldischargestatus
0    91.358025
1     8.641975
Name: proportion, dtype: float64


In [161]:
import joblib
'''
joblib.dump(X_train_processed, "X_train_processed.pkl")
joblib.dump(X_test_processed, "X_test_processed.pkl")
joblib.dump(y_train, "y_train.pkl")
joblib.dump(y_test, "y_test.pkl")
'''

joblib.dump(X_train_processed, "../model_dir/artifacts_dir/X_train_processed.pkl")
joblib.dump(X_test_processed, "../model_dir/artifacts_dir/X_test_processed.pkl")
joblib.dump(y_train, "../model_dir/artifacts_dir/y_train.pkl")
joblib.dump(y_test, "../model_dir/artifacts_dir/y_test.pkl")

['../model_dir/artifacts_dir/y_test.pkl']

In [162]:
import joblib

#joblib.dump(preprocessor, "preprocessor.pkl")

joblib.dump(preprocessor, "../model_dir/preprocessor.pkl")



['../model_dir/preprocessor.pkl']

In [163]:
import joblib

#joblib.dump(X_test, "X_test_raw.pkl")

joblib.dump(X_test, "../model_dir/artifacts_dir/X_test_raw.pkl")

['../model_dir/artifacts_dir/X_test_raw.pkl']

In [164]:
print(X_test.shape)

print(X_test_processed.shape)
print("----------------------------------------------")
print(X_test.iloc[0])

(486, 572)
(486, 830)
----------------------------------------------
patienthealthsystemstayid         1641313
gender                               Male
age                                  76.0
ethnicity                       Caucasian
hospitalid                            307
                                  ...    
troponin - T_std                      NaN
urinary creatinine_std                NaN
urinary osmolality_std                NaN
urinary sodium_std                    NaN
urinary specific gravity_std          NaN
Name: 1483, Length: 572, dtype: object


NOTE:

Your preprocessor was trained on roughly 572 raw input columns, which became 830 processed features.
 So a new patient input must contain the same expected raw columns.

That means we should not manually type 572 values.